# Gemma-2-2B + Gemma Scope: frontier-SAE difficulty test (Step 3 / Limitation 5)

Runs the **exact same** `extract_gemma.py` + `probe.py --sae_codes` from the repo on a GPU,
where Gemma-2-2B extraction takes minutes instead of ~35 h on a 16 GB Mac (where wired MPS
memory thrashes). The question: does an **off-the-shelf frontier-quality SAE** (Gemma Scope)
on a stronger model recover a difficulty signal that raw activations lack? If Δ(SAE−Raw) is
not significantly positive here, the null is not a small-model-SAE-quality artifact.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 is enough).
2. Accept the license at https://huggingface.co/google/gemma-2-2b (one click).
3. Have a HF token ready (https://huggingface.co/settings/tokens).

In [ ]:
# 1. Clone the repo and install deps
!git clone https://github.com/nabindev3/llm-sae-difficulty.git
%cd llm-sae-difficulty
!pip install -q sae_lens transformer_lens scikit-learn
!pip install -q -e . --no-deps

In [ ]:
# 2. Authenticate with Hugging Face (paste a read token; gemma-2-2b license must be accepted)
from huggingface_hub import login
login()

In [ ]:
# 3. Extract Gemma-2-2B activations + Gemma Scope codes (GPU: bf16, ~minutes for 5000)
!python extract_gemma.py --layer 12 --width 16k --max_samples 5000 --output_dir activations_gemma

In [ ]:
# 4. Probe raw Gemma activations vs Gemma Scope SAE codes (identical ladder via --sae_codes)
!python probing/probe.py \
  --activations activations_gemma/gemma_activations.safetensors \
  --metadata    activations_gemma/gemma_metadata.parquet \
  --sae_codes   activations_gemma/gemma_sae_codes.safetensors \
  --results_json activations_gemma/gemma_probe_results.json \
  --scores_parquet activations_gemma/gemma_scores.parquet

In [ ]:
# 5. Verdict: is Gemma Scope (frontier SAE) any better than raw on Gemma's own difficulty?
import json
r = json.load(open('activations_gemma/gemma_probe_results.json'))
print(f"P4 RawOnly AUROC : {r['P4_RawOnly_AUROC']:.4f}")
print(f"P5 SAEOnly AUROC : {r['P5_SAEOnly_AUROC']:.4f}")
print(f"Δ(SAE−Raw)        : {r['delta_sae_over_raw']:+.4f}  "
      f"95% CI [{r['delta_sae_over_raw_CI_lower']:+.4f}, {r['delta_sae_over_raw_CI_upper']:+.4f}]")
verdict = 'SAE BEATS raw (CI>0)' if r['delta_sae_over_raw_CI_lower'] > 0 else \
          'null holds: SAE does NOT beat raw'
print('=>', verdict)

In [ ]:
# 6. Download results to commit back into the repo (small JSON + metadata; codes optional)
from google.colab import files
files.download('activations_gemma/gemma_probe_results.json')
files.download('activations_gemma/gemma_metadata.parquet')
# Uncomment to also pull the tensors for local re-probing:
# files.download('activations_gemma/gemma_sae_codes.safetensors')
# files.download('activations_gemma/gemma_activations.safetensors')